# Evaluation — EN ArXiv

Full evaluation on ArXiv: generation + ROUGE + BERTScore + KIR +
abstraction metrics + LLM-as-Judge.


In [ ]:
import warnings, os, logging
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

try:
    import transformers, datasets
    transformers.utils.logging.set_verbosity_error()
    transformers.utils.logging.disable_progress_bar()
    datasets.utils.logging.set_verbosity_error()
    datasets.utils.logging.disable_progress_bar()
    logging.getLogger('sentence_transformers').setLevel(logging.ERROR)
except ImportError: pass

!uv pip install -e ../..
load_dotenv()
from huggingface_hub import login
login(token=os.getenv('HF_TOKEN'), add_to_git_credential=False)


In [ ]:
from sm_sip.config import SigExtConfig
from sm_sip.data import get_test_data
from sm_sip.models import load_sigext_model, unload_sigext_model, load_llm, create_summary_chain, create_judge_chain, preprocess_dataset
from sm_sip.prompts import get_summary_prompt, get_judge_prompt
from sm_sip.pipelines import run_enhanced_evaluation
from sm_sip.utils.io import save_results
from sm_sip.utils.gpu import clear_gpu_memory

sc = SigExtConfig.from_preset('en', 'xlmr-5k-060t')
data = get_test_data(lang='en', num_samples=100, skip_samples=sc.skip_samples)
sm, st = load_sigext_model(sc.model_id, device='cpu')
proc = preprocess_dataset(data, sm, st, lang='en')
unload_sigext_model(sm, st)

_, _, pipe = load_llm('meta-llama/Llama-3.1-8B-Instruct', '8bit')
chain = create_summary_chain(pipe, get_summary_prompt('en', 'source_aware'))
jchain = create_judge_chain(pipe, get_judge_prompt('en'))
metrics, samples = run_enhanced_evaluation(proc, chain, judge_chain=jchain, lang='en')

save_results({'metrics': metrics, 'samples': samples}, 'results/english/evaluation.json')
del pipe
clear_gpu_memory()
print('Done!')
